# AIML ZG519 – NLP Applications
## Assignment 2 – Programming Assignment: Information Extraction (10 Marks)

**File name:** `Assignment2_Programming_Group_<GroupNo>.ipynb`

Keep every section heading in this template. Each part needs code, executed outputs and a short markdown explanation.
Unexecuted or error-producing cells get no credit.

## Group details

| Field | Value |
| --- | --- |
| Group number | |
| Problem statement (PS1–PS7) | |
| Dataset | |

| # | Member name | BITS ID | Contribution |
| --- | --- | --- | --- |
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

## Problem statement and objective

*Restate your PS business context in 3–4 sentences, and the objective of your work.*

## Approach overview

*Summarise your pipeline: data preparation → CRF → BiLSTM → transformer → LLM → relation extraction → comparison.*

## 0. Setup

In [ ]:
# Install what you need (Colab). Add packages as required.
!pip install -q datasets seqeval sklearn-crfsuite transformers accelerate spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import random, json, re
import numpy as np
import pandas as pd

SEED = 42          # keep one seed for every random step and state it
random.seed(SEED)
np.random.seed(SEED)

## Part A – Data preparation and EDA (1.0)

Load your PS dataset, convert span annotations to BIO tags, and report:
- sentence and token counts per split
- entity-type distribution
- average entity length in tokens
- 3 example sentences printed with their BIO tags

See the dataset notes in your PS brief (inclusive vs exclusive end indices, character offsets for PS2).

In [ ]:
from datasets import load_dataset

# TODO: load your dataset as shown in your PS brief
# ds = load_dataset(...)

In [ ]:
def spans_to_bio(n_tokens, spans):
    """spans: list of (start, end_exclusive, type) in TOKEN indices.
    Returns a list of BIO tags. Convert inclusive ends (CrossRE) to exclusive before calling."""
    tags = ["O"] * n_tokens
    for start, end, etype in spans:
        tags[start] = f"B-{etype}"
        for i in range(start + 1, end):
            tags[i] = f"I-{etype}"
    return tags

# Sanity check
print(spans_to_bio(6, [(0, 2, "PER"), (4, 5, "LOC")]))

In [ ]:
# TODO: build train / validation / test lists of (tokens, bio_tags[, relations])
# TODO: EDA – counts per split, entity-type distribution, average entity length
# TODO: print 3 tagged example sentences

*Explain your data preparation decisions and what the EDA shows (class imbalance, rare types, sentence length).*

## Part B – Classical NER: CRF (1.5)

Train a CRF with hand-crafted features (word shape, case, prefix/suffix, POS, neighbouring words). Justify the features.
Report entity-level P/R/F1 per type with seqeval.

In [ ]:
import sklearn_crfsuite
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

def word2features(sent, i):
    # TODO: design your features
    w = sent[i]
    feats = {"bias": 1.0, "word.lower": w.lower(), "word.istitle": w.istitle()}
    return feats

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

# TODO: train the CRF and predict on the test split

In [ ]:
# TODO: print(classification_report(y_test, y_pred_crf, digits=4))

*Discuss which features helped and which entity types the CRF handles poorly.*

## Part C – Deep learning NER: BiLSTM / BiLSTM-CRF (1.5)

Use pretrained word embeddings (GloVe or similar). Plot training and validation loss. Report seqeval P/R/F1 per type.

In [ ]:
# TODO: build vocabulary, load pretrained embeddings, define BiLSTM(-CRF), train

In [ ]:
# TODO: plot training vs validation loss
# TODO: seqeval classification_report on the test split

*Explain your architecture choices and compare against the CRF.*

## Part D – Transformer fine-tuning (2.0)

Fine-tune a BERT-family model for token classification (suggested model in your PS brief).
**Subword-to-word label alignment is mandatory and must be explained.** A helper is given below; you may adapt it.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-cased"   # TODO: set the model for your PS
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(tokens, word_labels, label2id):
    """Label only the first sub-token of each word; other sub-tokens and special tokens get -100
    so the loss ignores them."""
    enc = tokenizer(tokens, is_split_into_words=True, truncation=True)
    labels, prev = [], None
    for wid in enc.word_ids():
        if wid is None:
            labels.append(-100)
        elif wid != prev:
            labels.append(label2id[word_labels[wid]])
        else:
            labels.append(-100)
        prev = wid
    enc["labels"] = labels
    return enc

# Show the alignment on one example sentence
ex_tokens = ["Aspirin", "caused", "gastrointestinal", "bleeding", "."]
ex_labels = ["B-DRUG", "O", "B-EFFECT", "I-EFFECT", "O"]
ex_map = {l: i for i, l in enumerate(sorted(set(ex_labels)))}
ex = tokenize_and_align(ex_tokens, ex_labels, ex_map)
print(list(zip(tokenizer.convert_ids_to_tokens(ex["input_ids"]), ex["labels"])))

In [ ]:
# TODO: build HF datasets, train with Trainer (AutoModelForTokenClassification), evaluate
# TODO: convert predictions back to word level before seqeval

*Explain the alignment, training settings and results.*

## Part E – LLM-based NER (1.5)

Run **zero-shot** and **few-shot (3–5 examples)** prompting on a fixed sample of at least 100 test sentences.
The LLM must return JSON. Map it back to token spans and score with seqeval.
Save prompts and raw responses to CSV, show the first 5, and count unparseable responses.

A simulated or hard-coded stand-in presented as an LLM scores zero. If you have no API access, use a small local open model and name it.

In [ ]:
SAMPLE_SIZE = 100
rng = random.Random(SEED)
# TODO: sample_idx = sorted(rng.sample(range(len(test_tokens)), SAMPLE_SIZE))

LLM_NAME = ""   # TODO: name of the API model or local model you used

In [ ]:
def llm_json_to_bio(tokens, raw_response, entity_types):
    """Parse the LLM's JSON list of {"text":..., "type":...} and map each entity to the first
    matching token span not already tagged. Returns (bio_tags, parsed_ok)."""
    tags = ["O"] * len(tokens)
    try:
        match = re.search(r"\[.*\]", raw_response, re.S)
        if match is None:
            return tags, False
        items = json.loads(match.group(0))
    except Exception:
        return tags, False
    lowered = [t.lower() for t in tokens]
    for item in items:
        etype = str(item.get("type", "")).strip()
        ent = str(item.get("text", "")).lower().split()
        if etype not in entity_types or not ent:
            continue
        for s in range(len(tokens) - len(ent) + 1):
            if lowered[s:s + len(ent)] == ent and all(t == "O" for t in tags[s:s + len(ent)]):
                tags[s] = f"B-{etype}"
                for k in range(s + 1, s + len(ent)):
                    tags[k] = f"I-{etype}"
                break
    return tags, True

# TODO: build zero-shot and few-shot prompts, call the LLM, keep raw responses
# TODO: save to CSV, e.g. pd.DataFrame(records).to_csv("llm_ner_outputs.csv", index=False)
# TODO: report parse failures and seqeval scores for both settings

*Compare zero-shot and few-shot. Discuss parse failures, cost and latency.*

## Part F – Relation extraction, light (1.5)

Candidates: all ordered pairs of gold entities in a sentence. Label = annotated relation, else `NO_RELATION`.
Cap `NO_RELATION` at twice the positive pairs in train and in test (use `SEED`).
CrossRE groups (PS3–PS7): keep the 5 most frequent relation types, map the rest to `OTHER`.

Method 1: classical baseline (e.g. logistic regression on entity types, words between, distance).
Method 2: LLM prompting **or** a fine-tuned transformer. Report macro-F1 and a confusion matrix for both.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report as cls_report, confusion_matrix, f1_score as sk_f1

# TODO: build candidate pairs and labels for train and test
# TODO: Method 1 – features + LogisticRegression, macro-F1, confusion matrix

In [ ]:
# TODO: Method 2 – LLM prompting or fine-tuned transformer, macro-F1, confusion matrix

*Which relations are hard, and why? How did the two methods differ?*

## Part G – Comparison, error analysis and reflection (1.0)

The table below must be built from variables computed above, on the **same** sample used in Part E.
Hand-typed numbers score zero for this part.

In [ ]:
# TODO: fill y_true_sample and each model's predictions on the sample indices
results = []
# for name, preds in [("CRF", crf_sample), ("BiLSTM", lstm_sample), ("Transformer", bert_sample),
#                     ("LLM zero-shot", llm0_sample), ("LLM few-shot", llmk_sample)]:
#     results.append({"Model": name,
#                     "Precision": precision_score(y_true_sample, preds),
#                     "Recall": recall_score(y_true_sample, preds),
#                     "F1": f1_score(y_true_sample, preds)})
comparison = pd.DataFrame(results)
comparison

In [ ]:
# TODO: show 10 errors from your best model with tokens, gold tags and predicted tags

### Error analysis
*Group the 10 errors: boundary errors, type confusion, missed rare types, others.*

### Reflection
*Which approach should the organisation in your PS use, and why? Cover accuracy, cost, data size and domain.
Every claim must match the numbers printed above.*

## References and acknowledgements

*List datasets, models, libraries, prompts, code sources and any AI tools used.*